<a href="https://colab.research.google.com/github/ZahiaYanes/medmcqa-lora-expert-fusion/blob/main/notebooks/02_lora_expert_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — LoRA Expert Training

This notebook trains domain-specific LoRA adapters on top of the frozen
Qwen2.5-1.5B-Instruct model. Each adapter specializes the model on one medical
subject (Pharmacology, Pathology, Surgery) without modifying the base weights.

Baseline accuracy to beat (from notebook 01, 50 validation questions per domain):

| Domain       | Baseline |
|--------------|----------|
| Pharmacology | 52%      |
| Pathology    | 46%      |
| Surgery      | 38%      |

## Reproducibility Note

This notebook documents an iterative process: an initial failed run, its diagnosis,
and the corrected approach. Re-running every training cell takes ~50 minutes of GPU.

Two flags control re-execution:

- `RERUN_HISTORICAL` — the narrative runs (failed training, diagnostics, small-scale
  validation). Their outputs are preserved in the saved notebook.
- `RETRAIN_EXPERTS` — the actual expert training and adapter saving. Keep this `False`
  unless you want to regenerate the adapters, as it overwrites files on Drive.

Set both to `True` to reproduce everything from scratch.

In [1]:
# Re-execution control (see note above)
RERUN_HISTORICAL = False   # failed run, diagnostics, 2k-example runs
RETRAIN_EXPERTS  = False   # full expert training + saving adapters to Drive
RERUN_CROSS_EVAL  = False   # cross-domain evaluation (~37 min)

# Quick check: is the working state still in memory?
try:
    dataset; tokenizer; lora_config; data_collator; train_expert
    print("State is loaded — new cells can be run directly.")
except NameError as e:
    print(f"Session was reset — re-run the state cells first ({e})")

Session was reset — re-run the state cells first (name 'dataset' is not defined)


In [2]:
# Colab ships an outdated torchao that is incompatible with recent peft versions.
# After running this cell: Runtime > Restart session, then run all cells again.
!pip install -U datasets transformers peft accelerate bitsandbytes torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.2 MB/s eta 0:00:00


In [3]:
import torch

# Fail fast: LoRA fine-tuning on CPU is impractically slow
assert torch.cuda.is_available(), "No GPU detected! Runtime > Change runtime type > T4 GPU"
print("Training on:", torch.cuda.get_device_name(0))

Training on: Tesla T4


## Loading Data and Base Model

Each Colab session is independent, so we reload the MedMCQA dataset and the base
model from scratch here.

In [4]:
from datasets import load_dataset

dataset = load_dataset("openlifescienceai/medmcqa")
print(dataset)

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # half precision to fit comfortably on a T4
    device_map="auto"
)

# Qwen does not define a padding token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


## Formatting Training Examples

Each question is converted into a single training string: the instruction prompt
followed by the correct answer letter.

The prompt is wrapped in the model's **chat template**
(`<|im_start|>user ... <|im_end|>`), which is the exact format Qwen2.5-Instruct
expects at inference time. Training and evaluation must share the same input
format, otherwise the model is optimized for a distribution it never sees.

In [6]:
LETTER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}

def build_prompt(example):
    # Instruction prompt, identical to the one used for baseline evaluation
    return f"""Answer the following medical question by responding with only the letter of the correct option (A, B, C, or D).

Question: {example['question']}
A) {example['opa']}
B) {example['opb']}
C) {example['opc']}
D) {example['opd']}

Answer:"""

def format_training_example(example):
    messages = [{"role": "user", "content": build_prompt(example)}]

    # add_generation_prompt=True appends the assistant turn header,
    # so the text ends exactly where the model would start generating
    templated = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    return {"text": templated + LETTER_MAP[example["cop"]]}

In [7]:
SUBJECT = "Pharmacology"

pharma_train = dataset["train"].filter(lambda x: x["subject_name"] == SUBJECT)
print(f"{SUBJECT} training examples: {len(pharma_train)}")

pharma_formatted = pharma_train.map(format_training_example)

# Inspect the exact string, special tokens included
print(repr(pharma_formatted[0]["text"]))

Filter:   0%|          | 0/182822 [00:00<?, ? examples/s]

Pharmacology training examples: 13758


Map:   0%|          | 0/13758 [00:00<?, ? examples/s]

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nAnswer the following medical question by responding with only the letter of the correct option (A, B, C, or D).\n\nQuestion: With which of the following receptors theophylline has an antagonistic interaction ?\nA) Histamine receptors\nB) Bradykinin receptors\nC) Adenosine receptors\nD) Imidazoline receptors\n\nAnswer:<|im_end|>\n<|im_start|>assistant\nC'


## Tokenization

LLMs process **tokens** (integer IDs), not raw text. We append an EOS token so the
model learns *when to stop* after producing the answer letter, and truncate at 256
tokens, which covers the vast majority of MedMCQA questions.

In [8]:
MAX_LENGTH = 256

def tokenize_function(example):
    text = example["text"] + tokenizer.eos_token
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,   # dynamic padding handled later by the data collator
    )

pharma_tokenized = pharma_formatted.map(
    tokenize_function,
    remove_columns=pharma_formatted.column_names   # keep only token IDs
)

print(pharma_tokenized)
print("Decoded tail:", tokenizer.decode(pharma_tokenized[0]["input_ids"])[-50:])

Map:   0%|          | 0/13758 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 13758
})
Decoded tail: nswer:<|im_end|>
<|im_start|>assistant
C<|im_end|>


## LoRA Configuration

LoRA freezes the base model and injects small trainable matrices into the attention
layers. Only those matrices are updated during training.

| Parameter | Value | Meaning |
|---|---|---|
| `r` | 16 | Rank of the low-rank bottleneck (capacity of the adapter) |
| `lora_alpha` | 32 | Scaling factor applied to the adapter output |
| `lora_dropout` | 0.05 | Regularization during training |
| `target_modules` | attention projections | Which layers receive adapters |

**Critical:** all three experts must use this exact same configuration. Merging
adapters by averaging their weights is only valid if they share identical shapes
and target the same layers.

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## Pipeline Validation Run

Before committing to full training (~13.7k examples), we train on a 2,000-example
subset to verify the pipeline runs end-to-end and that the loss decreases.

Note on the loss: it is computed over **all** tokens, including the prompt. Since
only one token per example is the actual answer, the loss plateaus at a relatively
high value — most of it comes from the inherently unpredictable question text.
This is addressed later with prompt masking.

In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

pharma_subset = pharma_tokenized.shuffle(seed=42).select(range(2000))

# mlm=False -> causal language modeling; labels are built from input_ids
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/checkpoints/pharmacology",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size = 16
    learning_rate=2e-4,              # LoRA tolerates higher LR than full fine-tuning
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

peft_model.config.use_cache = False   # disable KV cache during training

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=pharma_subset,
    data_collator=data_collator,
)

if RERUN_HISTORICAL:
    trainer.train()
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


In [11]:
def extract_letter(model_output):
    model_output = model_output.strip().upper()
    for letter in ["A", "B", "C", "D"]:
        if model_output.startswith(letter):
            return letter
    return None

def evaluate_model(eval_model, subject_name, n_samples=50):
    eval_model.eval()                      # inference mode (disables dropout)
    eval_model.config.use_cache = True     # re-enable KV cache for faster generation

    subject_data = [ex for ex in dataset["validation"] if ex["subject_name"] == subject_name]
    subject_data = subject_data[:n_samples]

    correct, total = 0, 0
    for example in subject_data:
        messages = [{"role": "user", "content": build_prompt(example)}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(eval_model.device)

        with torch.no_grad():
            outputs = eval_model.generate(**inputs, max_new_tokens=5, do_sample=False)

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        predicted = extract_letter(response)

        if predicted is not None:
            total += 1
            if predicted == LETTER_MAP[example["cop"]]:
                correct += 1

    return (correct / total if total else 0), correct, total

In [12]:
if RERUN_HISTORICAL:
    acc, correct, total = evaluate_model(peft_model, SUBJECT, n_samples=50)
    print(f"{SUBJECT} (2k-example adapter): {acc:.2%} ({correct}/{total})")
    print(f"Baseline reference:            52.00% (26/50)")
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


## Diagnosing the Failed Run

Accuracy dropped from 52% to 36% after fine-tuning. Since fine-tuning on in-domain
data should not *hurt* performance, we investigate rather than tweak
hyperparameters blindly.

We inspect two things:
- **Prediction distribution** — is the model collapsing onto one answer letter?
- **Base model under the same evaluation function** — using PEFT's
  `disable_adapter()` context manager, we can query the frozen base model in this
  exact session. If it reproduces the 52% from notebook 01, our evaluation is
  consistent and the regression genuinely comes from training.

In [13]:
from collections import Counter

def evaluate_with_diagnostics(eval_model, subject_name, n_samples=50):
    eval_model.eval()
    eval_model.config.use_cache = True

    subject_data = [ex for ex in dataset["validation"] if ex["subject_name"] == subject_name]
    subject_data = subject_data[:n_samples]

    predictions, expected_answers = [], []
    unparsed = 0

    for example in subject_data:
        messages = [{"role": "user", "content": build_prompt(example)}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(eval_model.device)

        with torch.no_grad():
            outputs = eval_model.generate(**inputs, max_new_tokens=5, do_sample=False)

        raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(raw)

        if pred is None:
            unparsed += 1
        else:
            predictions.append(pred)
            expected_answers.append(LETTER_MAP[example["cop"]])

    correct = sum(p == e for p, e in zip(predictions, expected_answers))
    acc = correct / len(predictions) if predictions else 0

    print(f"Accuracy: {acc:.2%} ({correct}/{len(predictions)})")
    print(f"Unparsed outputs: {unparsed}")
    print(f"Predicted letters : {dict(sorted(Counter(predictions).items()))}")
    print(f"Expected letters  : {dict(sorted(Counter(expected_answers).items()))}")

    return acc

In [14]:
if RERUN_HISTORICAL:
    print("=== FINE-TUNED ADAPTER ===")
    evaluate_with_diagnostics(peft_model, SUBJECT, n_samples=50)

    print("\n=== BASE MODEL (adapter disabled) ===")
    with peft_model.disable_adapter():
        evaluate_with_diagnostics(peft_model, SUBJECT, n_samples=50)
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


### Diagnostic Results

| Letter | Expected | Base model | Fine-tuned |
|--------|----------|-----------|------------|
| A | 14 | 20 | **29** |
| B | 14 | 10 | 6 |
| C | 15 | 13 | 11 |
| D | 7 | 7 | 4 |

The base model scores exactly 52.00%, matching notebook 01 — our evaluation is
consistent, so the regression is real.

The fine-tuned model answers "A" for 58% of questions, while only 28% of correct
answers are "A". It has learned the **prior distribution of answers** instead of
the medical content. No unparsable outputs, so this is not a formatting failure.

## Prompt Masking

Our first training run **degraded** performance: 36% vs 52% baseline. Diagnostics
revealed why — the fine-tuned model predicted "A" for 29/50 questions (58%), while
only 28% of correct answers are "A". It had learned the *prior distribution* of
answers rather than the medical content.

**Root cause:** the loss was computed over all ~100 tokens of each example, but
only **one** token is the actual answer. The learning signal on that token was
diluted by 99% of gradient spent on reproducing question text — which is both
useless and inherently unpredictable.

**Fix:** mask the prompt by setting its labels to `-100`, a sentinel value that
PyTorch's cross-entropy loss ignores. All gradient then focuses on predicting the
answer letter.

Expect the reported loss to *increase* after this change — it now measures only
the answer token, which is a genuinely harder prediction, instead of being
averaged down by easy-to-predict prompt tokens.

In [15]:
def tokenize_with_masking(example):
    # Build prompt and answer separately so we can mask the prompt portion
    messages = [{"role": "user", "content": build_prompt(example)}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    answer_text = LETTER_MAP[example["cop"]] + tokenizer.eos_token

    # add_special_tokens=False: the chat template already includes them
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + answer_ids

    # -100 tells the loss function to skip these positions
    labels = [-100] * len(prompt_ids) + answer_ids

    # Truncate from the left if needed, so the answer is never cut off
    if len(input_ids) > MAX_LENGTH:
        overflow = len(input_ids) - MAX_LENGTH
        input_ids = input_ids[overflow:]
        labels = labels[overflow:]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


# Apply to the raw dataset (not the pre-formatted one)
pharma_masked = pharma_train.map(
    tokenize_with_masking,
    remove_columns=pharma_train.column_names
)

# Verify: only the final tokens should have real labels
sample = pharma_masked[0]
print("Total tokens:", len(sample["labels"]))
print("Unmasked labels:", [l for l in sample["labels"] if l != -100])
print("Decoded unmasked:", tokenizer.decode([l for l in sample["labels"] if l != -100]))

Map:   0%|          | 0/13758 [00:00<?, ? examples/s]

Total tokens: 104
Unmasked labels: [34, 151645]
Decoded unmasked: C<|im_end|>


### Why the Data Collator Must Change

`DataCollatorForLanguageModeling` builds labels by copying `input_ids`, which would
silently overwrite the `-100` masks we just created.

`DataCollatorForSeq2Seq` preserves the `labels` field as-is and pads it with
`-100`, which is what we need here.

In [16]:
from transformers import DataCollatorForSeq2Seq

# DataCollatorForLanguageModeling would overwrite our labels from input_ids.
# DataCollatorForSeq2Seq preserves them and pads labels with -100.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

In [17]:
if RERUN_HISTORICAL:
    # The current peft_model carries the degraded adapter — reload from scratch
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    peft_model = get_peft_model(model, lora_config)
    peft_model.config.use_cache = False
    peft_model.print_trainable_parameters()
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


In [18]:
if RERUN_HISTORICAL:
    pharma_subset = pharma_masked.shuffle(seed=42).select(range(2000))

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=pharma_subset,
        data_collator=data_collator,
    )

    trainer.train()
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


In [19]:
if RERUN_HISTORICAL:
    print("=== MASKED FINE-TUNING ===")
    evaluate_with_diagnostics(peft_model, SUBJECT, n_samples=50)
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


### Masked Run (2,000 examples)

| Model | Accuracy | "A" predictions |
|---|---|---|
| No masking | 36% | 29/50 |
| With masking | 54% | 24/50 |
| Base model | 52% | 20/50 |

The collapse is corrected — predictions are far better distributed. However, the
+2 point gap over baseline is **not meaningful**: with n=50, the 95% margin of
error is roughly ±14 points.

A note on the reported loss, which *dropped* from ~0.68 to ~0.50: this is not a
direct improvement signal. Only two tokens are now unmasked (the answer letter and
EOS), and EOS is trivially predictable after an isolated letter. The average is
therefore roughly half the true difficulty — the real loss on the answer token is
closer to 1.0, versus ln(4) ≈ 1.386 for random guessing.

**Loss values are only interpretable relative to what they measure.** To draw
conclusions, we scale up both training data and evaluation size.

## Scaling Up

Two changes to obtain an interpretable result:

| | Before | Now |
|---|---|---|
| Training examples | 2,000 | **13,758** (full Pharmacology set) |
| Validation questions | 50 | **243** (full Pharmacology validation) |

With n=243, the margin of error drops to roughly ±6 points, making a 5+ point gap
interpretable.

In [20]:
if RETRAIN_EXPERTS:
    peft_model.config.use_cache = False
    peft_model.train()

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=pharma_masked,   # full dataset, no subset
        data_collator=data_collator,
    )

    trainer.train()
else:
    print("Skipped — adapter already trained and saved to Drive.")

Skipped — adapter already trained and saved to Drive.


In [21]:
if RETRAIN_EXPERTS:
    print("=== FULL TRAINING, FULL VALIDATION ===")
    evaluate_with_diagnostics(peft_model, SUBJECT, n_samples=243)

    print("\n=== BASE MODEL, FULL VALIDATION ===")
    with peft_model.disable_adapter():
        evaluate_with_diagnostics(peft_model, SUBJECT, n_samples=243)
else:
    print("Skipped — original output preserved above.")

Skipped — original output preserved above.


## Results: Pharmacology Expert

Trained on the full Pharmacology training set (13,758 examples, 1 epoch) with
prompt masking, evaluated on the full Pharmacology validation set (243 questions).

| Model | Accuracy | Correct |
|---|---|---|
| Base Qwen2.5-1.5B-Instruct | 50.21% | 122/243 |
| + Pharmacology LoRA adapter | **60.91%** | 148/243 |

A **+10.7 point** improvement. With n=243, the 95% margin of error on a single
accuracy estimate is roughly ±6 points, so this gap is well outside the noise
floor.

Answer distribution also improved: the base model over-predicted "A" (97 vs 78
expected) and under-predicted "B" (47 vs 72); the fine-tuned model is noticeably
better calibrated (90 and 63 respectively).

**Open question:** we cannot yet tell whether this gain reflects better use of
medical knowledge, or adaptation to MedMCQA's specific question style and answer
distribution. Evaluating on a held-out medical benchmark (MedQA, PubMedQA) would
disentangle the two. Noted as future work.


## Persisting the Adapter

Colab wipes its filesystem when a session ends, and the next training run will
overwrite `peft_model` in memory. We save the adapter to Google Drive before
moving on.

`save_pretrained()` on a PEFT model writes **only the adapter weights** (a few MB),
not the frozen base model (~3 GB) — the practical payoff of LoRA's parameter
efficiency.

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
import os

SAVE_ROOT = "/content/drive/MyDrive/medmcqa-experts"
os.makedirs(SAVE_ROOT, exist_ok=True)

pharma_path = f"{SAVE_ROOT}/pharmacology"

In [25]:
if RETRAIN_EXPERTS:
    # Saves only the LoRA adapter (~17 MB), not the full base model
    peft_model.save_pretrained(pharma_path)
    print("Saved to:", pharma_path)
    print("Files:", os.listdir(pharma_path))
else:
    print("Skipped — adapter already saved.")

Skipped — adapter already saved.


In [26]:
import os
size_mb = os.path.getsize(f"{pharma_path}/adapter_model.safetensors") / 1024**2
print(f"Adapter size: {size_mb:.1f} MB")

Adapter size: 16.7 MB


## Training the Remaining Experts

The Pharmacology pipeline is now validated, so we factor it into a reusable
function and apply it to Pathology and Surgery.

Each expert starts from a **freshly loaded base model and a freshly initialized
adapter** — they must be trained independently, never sequentially on top of one
another, otherwise the second adapter would inherit the first one's specialization.

All three share the identical `lora_config`. This is a hard requirement for the
merging step: averaging adapter weights is only mathematically valid if they have
the same rank and target the same modules.

In [27]:
def train_expert(subject, n_epochs=1):
    """Train a LoRA adapter specialized on one medical subject."""

    # Fresh base model + fresh adapter for each expert
    base = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    expert = get_peft_model(base, lora_config)
    expert.config.use_cache = False
    expert.train()

    # Prepare this subject's training data with prompt masking
    subject_data = dataset["train"].filter(lambda x: x["subject_name"] == subject)
    subject_tokenized = subject_data.map(
        tokenize_with_masking,
        remove_columns=subject_data.column_names
    )
    print(f"{subject}: {len(subject_tokenized)} training examples")

    args = TrainingArguments(
        output_dir=f"outputs/checkpoints/{subject.lower()}",
        num_train_epochs=n_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )

    Trainer(
        model=expert,
        args=args,
        train_dataset=subject_tokenized,
        data_collator=data_collator,
    ).train()

    # Persist the adapter before moving on
    path = f"{SAVE_ROOT}/{subject.lower()}"
    expert.save_pretrained(path)
    print("Saved:", path)

    return expert

In [28]:
if RETRAIN_EXPERTS:
    pathology_expert = train_expert("Pathology")
else:
    print("Skipped — adapter already trained and saved to Drive.")

Skipped — adapter already trained and saved to Drive.


In [29]:
if RETRAIN_EXPERTS:
    surgery_expert = train_expert("Surgery")
else:
    print("Skipped — adapter already trained and saved to Drive.")

Skipped — adapter already trained and saved to Drive.


## Cross-Domain Evaluation

Each expert is evaluated on **all three domains**, not just its own. This measures
two distinct effects:

- **Specialization gain** — does the expert beat the base model on its own domain?
- **Cross-domain cost** — does specializing on one subject *degrade* the others?

This distinction is what motivates merging. If each expert is strong on its domain
but weak elsewhere, a merged model that performs well across all three would be
genuinely useful. If instead every expert improves every domain (positive transfer),
merging may be unnecessary — simply training on the pooled data would suffice.

Experts are reloaded from Drive one at a time rather than kept in memory, to avoid
exhausting the T4's VRAM.

In [30]:
import gc
from peft import PeftModel

EVAL_SIZES = {"Pharmacology": 243, "Pathology": 337, "Surgery": 369}

def load_expert(subject):
    """Load a saved LoRA adapter on top of a fresh base model."""
    base = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    return PeftModel.from_pretrained(base, f"{SAVE_ROOT}/{subject.lower()}")

def evaluate_across_domains(eval_model, label, n_per_domain=None):
    """Evaluate one model on all three domains."""
    results = {}
    for subject, full_size in EVAL_SIZES.items():
        n = n_per_domain or full_size
        acc, correct, total = evaluate_model(eval_model, subject, n_samples=n)
        results[subject] = acc
        print(f"  {label:12s} on {subject:14s}: {acc:.2%} ({correct}/{total})")
    return results

def free_memory(*objects):
    """Release GPU memory between model loads."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

In [31]:
if RERUN_CROSS_EVAL:
    N_EVAL = 150   # per domain; set to None for the full validation sets

    cross_results = {}

    # Base model reference (adapter disabled on a loaded expert)
    expert = load_expert("Pharmacology")
    print("=== BASE MODEL ===")
    with expert.disable_adapter():
        cross_results["Base"] = evaluate_across_domains(expert, "Base", N_EVAL)

    print("\n=== PHARMACOLOGY EXPERT ===")
    cross_results["Pharmacology"] = evaluate_across_domains(expert, "Pharma", N_EVAL)
    free_memory(expert)

    for subject in ["Pathology", "Surgery"]:
        expert = load_expert(subject)
        print(f"\n=== {subject.upper()} EXPERT ===")
        cross_results[subject] = evaluate_across_domains(expert, subject[:8], N_EVAL)
        free_memory(expert)
else:
    print("Skipped — original output preserved above.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== BASE MODEL ===
  Base         on Pharmacology  : 51.33% (77/150)
  Base         on Pathology     : 48.67% (73/150)
  Base         on Surgery       : 39.60% (59/149)

=== PHARMACOLOGY EXPERT ===
  Pharma       on Pharmacology  : 56.00% (84/150)
  Pharma       on Pathology     : 55.33% (83/150)
  Pharma       on Surgery       : 40.67% (61/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== PATHOLOGY EXPERT ===
  Patholog     on Pharmacology  : 52.00% (78/150)
  Patholog     on Pathology     : 63.33% (95/150)
  Patholog     on Surgery       : 46.67% (70/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== SURGERY EXPERT ===
  Surgery      on Pharmacology  : 53.33% (80/150)
  Surgery      on Pathology     : 57.33% (86/150)
  Surgery      on Surgery       : 48.67% (73/150)


In [32]:
if RERUN_CROSS_EVAL:
    import pandas as pd

    df_cross = pd.DataFrame(cross_results).T * 100
    df_cross.index.name = "Model"
    print(df_cross.round(2).to_string())
else:
    print("Skipped — original output preserved above.")

              Pharmacology  Pathology  Surgery
Model                                         
Base                 51.33      48.67    39.60
Pharmacology         56.00      55.33    40.67
Pathology            52.00      63.33    46.67
Surgery              53.33      57.33    48.67


## Summary

Three LoRA adapters were trained independently on the same frozen
Qwen2.5-1.5B-Instruct base, with identical hyperparameters (r=16, alpha=32,
attention projections only) — a prerequisite for weight-space merging.

| Expert | Training examples | Final loss | Adapter size |
|---|---|---|---|
| Pharmacology | 13,758 | 0.39 | ~17 MB |
| Pathology | 14,884 | 0.37 | ~17 MB |
| Surgery | 16,862 | 0.47 | ~17 MB |

### Cross-domain accuracy (150 validation questions per domain)

| Model | Pharmacology | Pathology | Surgery |
|---|---|---|---|
| Base | 51.33 | 48.67 | 39.60 |
| Pharmacology expert | **56.00** | 55.33 | 40.67 |
| Pathology expert | 52.00 | **63.33** | 46.67 |
| Surgery expert | 53.33 | 57.33 | **48.67** |

**Specialization works** — each expert is the strongest model on its own domain.

**But there is no cross-domain cost.** All nine off-diagonal cells improve on the
base model. We expected a trade-off (gains on the target domain, losses elsewhere);
instead we observe consistent **positive transfer**. Training on Pathology also
improves Surgery by +7.1 points.

This reframes the merging question. The original motivation — "each expert is
narrow, so merge them to get broad coverage" — does not hold. Notebook 03 must
therefore compare merging against a stronger baseline: a **single adapter trained
on all three domains pooled together**, which may already capture this transfer.

### Caveats

- With n=150 per domain, the 95% margin of error is roughly ±8 points. Only the
  Pathology (+14.7) and Surgery (+9.1) self-domain gains clearly exceed the noise
  floor.
- The validation set is not randomly ordered: the Pharmacology expert scores 56.0%
  on the first 150 questions but 60.9% on all 243. Sampling is randomized in the
  final evaluation.
- We cannot yet distinguish genuine medical knowledge gains from adaptation to
  MedMCQA's question style. Evaluating on a held-out benchmark (MedQA, PubMedQA)
  would disentangle the two.

Adapters are persisted to Drive and combined in notebook 03.